In [ ]:
from pathlib import Path
import re
from collections import defaultdict

DOCUMENTS_DIR = Path(__file__).parent / "documents"


def tokenize(text: str) -> list[str]:
    """Convert text to lowercase word tokens."""
    return re.findall(r"\b\w+\b", text.lower())


def build_inverted_index(folder: Path) -> dict[str, set[str]]:
    """Map each term to the names of text files that contain it."""
    index: defaultdict[str, set[str]] = defaultdict(set)
    for path in sorted(folder.glob("*.txt")):
        text = path.read_text(encoding="utf-8", errors="ignore")
        for term in set(tokenize(text)):
            index[term].add(path.name)
    return dict(index)


def search(index: dict[str, set[str]], query: str) -> list[str]:
    """Return files containing all query terms."""
    terms = tokenize(query)
    if not terms:
        return []

    matches = index.get(terms[0], set()).copy()
    for term in terms[1:]:
        matches.intersection_update(index.get(term, set()))
    return sorted(matches)


def main() -> None:
    DOCUMENTS_DIR.mkdir(exist_ok=True)
    index = build_inverted_index(DOCUMENTS_DIR)
    if not index:
        print(f"No .txt documents found in: {DOCUMENTS_DIR}")
        print("Add some text files to that folder, then run this program again.")
        return

    print(f"Indexed {len(index)} unique terms from {DOCUMENTS_DIR}")
    print("Type a search query, or 'quit' to exit.")
    while True:
        query = input("\nSearch: ").strip()
        if query.lower() in {"quit", "exit", "q"}:
            break
        results = search(index, query)
        if results:
            print("Matching documents:")
            for filename in results:
                print(f"- {filename}")
        else:
            print("No documents matched all search terms.")


if __name__ == "__main__":
    main()